In [ ]:
### 一些杂项
事务事实表中的status应该表示事件发生时的瞬时状态，比如下单时的状态，如 'created'，支付结果，如 'success'/'failed'，如果 status 会更新，就违反了事务事实表"只插入不更新"的原则
事务事实表只应该出现下单的信息，意味着不要因为业务更新

### 数仓中各层级的职责明细
ods: 原始数据层，和原始数据保持一致，除了create_time，update_time，etl_time之外，其它所有字段全部使用string，如果业务系统能保证数据类型不会变，或者确定不会出错，也可以和业务系统保持部分一致，比如int对应int，double对应double，decimal对应decimal，以减少存储成本同时提升计算效率，但我个人推荐使用全string
dwd：依据一个原则来设计
1. 字段规范化收口，因为ods层是原始数据，非常散乱，业务系统各个表字段名称随意，字段类型参差不齐，同时多源同名异意，因此需要dwd层对其规范统一，将字段名标准化，类型统一对齐，一指标一定义
2. 按业务过程拆表，所谓的业务过程就是一次不可拆分的业务动作，有明确的触发时间点，比如说下单过程，支付过程，退款过程。
3. 一个指标只归属一张表，同一原子指标在 DWD 层只出现一次，禁止跨表重复定义，比如说下单表的order_amt，支付表的pay_amt，退款表的refund_amt
4. 严格从 ODS 取数，禁止跨层
5. 只装原子指标和维度外键及一些维度退化的维度，派生指标和衍生指标不允许出现在这里
6. 粒度必须明确且唯一，粒度决定了主键是什么，能回答什么粒度的分析问题以及能否与维度表正确 JOIN，大部分粒度和ods层的主表保持一致，但这不是绝对的
dws：dws层是数仓的指标统一出口，其核心使命是把DWD层分散的原子指标组装成面向分析的派生指标，它也有几点规范
1. 所有面向业务的指标必须从 DWS 取，禁止绕过 DWS 直接拼 DWD，例如下游各自从订单表和支付表及退单表拼指标，导致口径各有差异
2. 面向主题域建表，比如交易域，流量域，活动域，用户域，商品域等
3. 派生指标=原子指标 + 修饰词 + 时间周期 + 聚合维度，例如pay_amt + 母婴品类 + 最近7天 + 用户粒度 = dws_trade_user_1di.mother_baby_pay_amt_7d，现在为了灵活也可以把时间周期合并进聚合维度中，使用biz_date指代日期，将时间周期固定在一天，这样能看到每个按天聚合的日期，不同限定在最近7天。
4. 粒度由维度组合决定，必须显式声明，比如
dws_trade_user_1di            粒度：user_id × date
dws_trade_user_product_1di    粒度：user_id × product_id × date
dws_trade_product_category_1di 粒度：product_id × category_id × date
dws_traffic_page_1di          粒度：page_id × date
5. 跨过程指标在 DWS 组装，DWD 不做
6. 时间粒度分层，不混存
dwa：面向业务建表

### 数仓中事实表分类
事务事实表：一行代表一个具体的业务操作，通常只插入，不更新，是最常用的
周期快照事实表：在固定时间间隔对业务状态进行快照，一行代表某个时间点（天/周/月）的汇总状态，示例：每日库存快照表、每月账户余额表、每天订单状态汇总表
累积快照事实表：跟踪业务实体的完整生命周期，记录关键里程碑，示例：订单生命周期表（包含下单时间、支付时间、发货时间、签收时间等）
无事实的事实表：只有维度外键，没有度量值，示例：用户签到表、活动参与记录表

### 数仓中维度表分类
